In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ─────────────────────────────────────────────────────
# 📍 Read Coordinates from INP File
# ─────────────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    in_coords = False
    for line in lines:
        if line.strip().upper() == '[COORDINATES]':
            in_coords = True
            continue
        if in_coords and line.strip().startswith('['):
            break
        if in_coords and line.strip() and not line.strip().startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ─────────────────────────────────────────────────────
# 🗺️ Compare and Plot Map per Tag
# ─────────────────────────────────────────────────────
def plot_flooding_comparison(inp_file, tag, coords, base_path):
    with_backup_path = base_path / "SWMM_Results" / "35-45 mph" / "with_back_up_power_southeast" / "basement_flooding_csv" / f"{tag}(35-45)_flooding.csv"
    without_backup_path = base_path / "SWMM_Results" / "35-45 mph" / "no_back_up_power" / "basement_flooding_csv" / f"{tag}nobackuppower(35-45)_flooding.csv"

    if not with_backup_path.exists() or not without_backup_path.exists():
        print(f"⚠️ Missing CSVs for tag '{tag}'")
        return None

    try:
        with_backup = pd.read_csv(with_backup_path)
        without_backup = pd.read_csv(without_backup_path)
    except Exception as e:
        print(f"❌ Error reading CSVs for tag '{tag}': {e}")
        return None

    if "Node" not in with_backup or "Node" not in without_backup:
        print(f"❌ Missing 'Node' column for tag '{tag}'")
        return None
    if "basement flooding" not in with_backup or "basement flooding" not in without_backup:
        print(f"❌ Missing 'basement flooding' column for tag '{tag}'")
        return None

    with_backup["Node"] = with_backup["Node"].astype(str)
    without_backup["Node"] = without_backup["Node"].astype(str)

    flooded_with = set(with_backup[with_backup["basement flooding"] == 1]["Node"])
    flooded_without = set(without_backup[without_backup["basement flooding"] == 1]["Node"])

    both = flooded_with & flooded_without
    only_with = flooded_with - flooded_without
    only_without = flooded_without - flooded_with

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in coords],
        y=[coords[n][1] for n in coords],
        mode='markers', marker=dict(color='lightgrey', size=4),
        name='All Nodes', hoverinfo='skip'))

    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in both if n in coords],
        y=[coords[n][1] for n in both if n in coords],
        mode='markers', marker=dict(color='red', size=4),
        name='Flooded (Both)', text=list(both), hoverinfo='text'))

    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in only_with if n in coords],
        y=[coords[n][1] for n in only_with if n in coords],
        mode='markers', marker=dict(color='green', size=4),
        name='Flooded (Only With Backup)', text=list(only_with), hoverinfo='text'))

    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in only_without if n in coords],
        y=[coords[n][1] for n in only_without if n in coords],
        mode='markers', marker=dict(color='blue', size=4),
        name='Flooded (Only Without Backup)', text=list(only_without), hoverinfo='text'))

    fig.update_layout(title=f"Flooding Comparison Map: {tag}",
                      height=700, width=950)

    output_map = base_path / f"{tag}_(70-80)comparison_flooding_map.html"
    fig.write_html(str(output_map))
    print(f"🧭 Saved map for tag '{tag}': {output_map.name}")

    return {
        "Tag": tag,
        "Flooded (Both)": len(both),
        "Only With Backup": len(only_with),
        "Only Without Backup": len(only_without)
    }

# ─────────────────────────────────────────────────────
# 🚀 Run for All Tags
# ─────────────────────────────────────────────────────
def run_flooding_comparison_all_tags(base_path_str, inp_file_name):
    base_path = Path(base_path_str)
    inp_file = base_path / inp_file_name
    coords = read_coordinates(inp_file)

    # Folder with the NO BACKUP POWER CSVs
    csv_dir = base_path / "SWMM_Results" / "35-45 mph" / "no_back_up_power" / "basement_flooding_csv"
    tags = [f.stem.replace("nobackuppower(35-45)_flooding", "") for f in csv_dir.glob("*nobackuppower(35-45)_flooding.csv")]
    
    if not tags:
        print("⚠️ No tags found. Check path or file naming.")
        return

    print(f"🔍 Found {len(tags)} tags: {tags}")
    
    summary = []
    for tag in tags:
        result = plot_flooding_comparison(inp_file, tag, coords, base_path)
        if result:
            summary.append(result)

    print(f"✅ Processed {len(summary)} of {len(tags)} tags.")

    summary_df = pd.DataFrame(summary)
    summary_csv = base_path / "flooding_comparison_summary35-45.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"📊 Summary saved: {summary_csv.name}")

# ─────────────────────────────────────────────────────
# 🔧 Edit Here Before Running
# ─────────────────────────────────────────────────────
if __name__ == "__main__":
    run_flooding_comparison_all_tags(
        base_path_str=r"C:\Users\csofios\Desktop\LongTermCSO\SWMM",
        inp_file_name="GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
    )


🔍 Found 80 tags: ['pump_status_100', 'pump_status_10', 'pump_status_11', 'pump_status_12', 'pump_status_13', 'pump_status_14', 'pump_status_15', 'pump_status_16', 'pump_status_17', 'pump_status_18', 'pump_status_19', 'pump_status_20', 'pump_status_21', 'pump_status_22', 'pump_status_23', 'pump_status_24', 'pump_status_25', 'pump_status_26', 'pump_status_27', 'pump_status_28', 'pump_status_29', 'pump_status_2', 'pump_status_30', 'pump_status_31', 'pump_status_32', 'pump_status_33', 'pump_status_34', 'pump_status_35', 'pump_status_36', 'pump_status_37', 'pump_status_38', 'pump_status_39', 'pump_status_3', 'pump_status_40', 'pump_status_41', 'pump_status_42', 'pump_status_43', 'pump_status_44', 'pump_status_45', 'pump_status_46', 'pump_status_47', 'pump_status_48', 'pump_status_49', 'pump_status_4', 'pump_status_50', 'pump_status_51', 'pump_status_52', 'pump_status_53', 'pump_status_54', 'pump_status_55', 'pump_status_56', 'pump_status_57', 'pump_status_58', 'pump_status_59', 'pump_status

In [3]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ───────────────────────────────────────────────
# 📍 Read Coordinates from INP File
# ───────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    in_coords = False
    for line in lines:
        line = line.strip()
        if line.upper() == '[COORDINATES]':
            in_coords = True
            continue
        if in_coords and line.startswith('['):
            break
        if in_coords and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ───────────────────────────────────────────────
# 🚀 Create Single Difference Map with Background
# ───────────────────────────────────────────────
def plot_flooding_difference_only(base_path_str, inp_file_name):
    base_path = Path(base_path_str)
    inp_file = base_path / inp_file_name
    coords = read_coordinates(inp_file)

    folder_with = base_path / "SWMM_Results" / "35-45 mph" / "with_back_up_power_southeast" / "basement_flooding_csv"
    folder_without = base_path / "SWMM_Results" / "35-45 mph" / "no_back_up_power" / "basement_flooding_csv"

    tags = [f.stem.replace("nobackuppower(35-45)_flooding", "") for f in folder_without.glob("*nobackuppower(35-45)_flooding.csv")]
    print(f"🔍 Found {len(tags)} scenarios")

    count_with, count_without = {}, {}
    total = len(tags)

    for tag in tags:
        try:
            df_with = pd.read_csv(folder_with / f"{tag}(35-45)_flooding.csv")
            df_without = pd.read_csv(folder_without / f"{tag}nobackuppower(35-45)_flooding.csv")

            for df, store in [(df_with, count_with), (df_without, count_without)]:
                flooded_nodes = df[df["basement flooding"] == 1]["Node"].astype(str)
                for node in flooded_nodes:
                    store[node] = store.get(node, 0) + 1
        except Exception as e:
            print(f"⚠️ Error processing tag '{tag}': {e}")
            continue

    # Build master dataframe
    all_nodes = set(coords.keys())
    df = pd.DataFrame({'Node': list(all_nodes)})
    df['x'] = df['Node'].map(lambda n: coords[n][0])
    df['y'] = df['Node'].map(lambda n: coords[n][1])
    df['FloodProb_With'] = df['Node'].map(lambda n: count_with.get(n, 0) / total)
    df['FloodProb_Without'] = df['Node'].map(lambda n: count_without.get(n, 0) / total)
    df['Difference'] = df['FloodProb_Without'] - df['FloodProb_With']

    # Filter nodes where flooding behavior differs
    df_diff = df[df['Difference'].abs() > 1e-1].copy()
    if df_diff.empty:
        print("ℹ️ No nodes with differing probabilities between the two scenarios.")
        return

    # ───────────────────────────────────────────────
    # 🗺️ Plot Map: Background + Differences Only
    # ───────────────────────────────────────────────
    fig = go.Figure()

    # ⚪ Background nodes (all)
    fig.add_trace(go.Scatter(
        x=df['x'],
        y=df['y'],
        mode='markers',
        marker=dict(color='lightgrey', size=4),
        hoverinfo='skip',
        showlegend=False
    ))

    # 🔵/🔴 Nodes with difference
    fig.add_trace(go.Scatter(
        x=df_diff['x'],
        y=df_diff['y'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_diff['Difference'],
            colorscale='RdBu',
            cmin=-1,
            cmax=1,
            colorbar=dict(
                title="Diff Flooding Probability (No Backup - With Backup)",
                tickvals=[-1, -0.5, 0, 0.5, 1],
                ticktext=["Much Worse With Backup", "-0.5", "No Change", "+0.5", "Much Worse Without Backup"]
            ),
            line=dict(width=0.5, color='black')
        ),
        text=df_diff["Node"] + "<br>Δ Flood Prob: " + (df_diff["Difference"] * 100).round(1).astype(str) + "%",
        hoverinfo="text",
        showlegend=False
    ))

    fig.update_layout(
        title="🧭 Difference in Basement Flooding Probability (No Backup - With Backup)",
        xaxis_title="X",
        yaxis_title="Y",
        height=800,
        width=1000,
        hovermode="closest",
        dragmode="zoom"
    )

    output_html = base_path / "flooding_difference_map_35-45.html"
    fig.write_html(str(output_html))
    print(f"\n✅ Difference-only map saved to: {output_html}")

# ───────────────────────────────────────────────
# 🔧 Run the Script
# ───────────────────────────────────────────────
if __name__ == "__main__":
    plot_flooding_difference_only(
        base_path_str=r"C:\Users\csofios\Desktop\LongTermCSO\SWMM",
        inp_file_name="GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
    )


🔍 Found 80 scenarios

✅ Difference-only map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\flooding_difference_map_35-45.html


In [ ]:
import pandas as pd
from pathlib import Path

def print_critical_nodes(base_path_str):
    base_path = Path(base_path_str)
    # Path to previously generated data (from main script)
    inp_file = base_path / "GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"

    # Rebuild flooding probability data from previous results
    folder_with = base_path / "SWMM_Results" / "50-60 mph" / "with_back_up_power_southeast" / "basement_flooding_csv"
    folder_without = base_path / "SWMM_Results" / "50-60 mph" / "no_back_up_power" / "basement_flooding_csv"

    # Extract tags (scenario identifiers)
    tags = [f.stem.replace("nobackuppower(50-60)_flooding", "") for f in folder_without.glob("*nobackuppower(50-60)_flooding.csv")]
    total = len(tags)

    count_with, count_without = {}, {}

    for tag in tags:
        try:
            df_with = pd.read_csv(folder_with / f"{tag}(50-60)_flooding.csv")
            df_without = pd.read_csv(folder_without / f"{tag}nobackuppower(50-60)_flooding.csv")

            for df, store in [(df_with, count_with), (df_without, count_without)]:
                flooded_nodes = df[df["basement flooding"] == 1]["Node"].astype(str)
                for node in flooded_nodes:
                    store[node] = store.get(node, 0) + 1
        except Exception as e:
            print(f"⚠️ Error processing tag '{tag}': {e}")
            continue

    all_nodes = set(count_with.keys()) | set(count_without.keys())

    df = pd.DataFrame({'Node': list(all_nodes)})
    df['FloodProb_With'] = df['Node'].map(lambda n: count_with.get(n, 0) / total)
    df['FloodProb_Without'] = df['Node'].map(lambda n: count_without.get(n, 0) / total)

    # Filter for critical nodes
    critical_nodes = df[(df['FloodProb_Without'] == 1.0) & (df['FloodProb_With'] == 0.0)]

    # Print and save
    if not critical_nodes.empty:
        print("\n🚨 Nodes with 100% flooding without backup and 0% with backup:")
        for node in critical_nodes['Node']:
            print(f" - {node}")

        output_csv = base_path / "critical_flood_nodes_50-60.csv"
        critical_nodes.to_csv(output_csv, index=False)
        print(f"\n✅ Saved critical node list to: {output_csv}")
    else:
        print("\n✅ No critical nodes found with 100% worse flooding without backup.")

# 🔧 Run this script
if __name__ == "__main__":
    print_critical_nodes(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM")
